In [1]:
# Simple urban morphology visualization
import folium
import geopandas as gpd
from shapely.geometry import box, Point

# Load data
grid = gpd.read_file("Data/caba_building_metrics.geojson")
buildings = gpd.read_file("Data/CABA_Buildings.geojson")
buildings = buildings[buildings["confidence"] >= 0.75]

# Create a 200×200m sample area around my location
lat, lon = -34.5865923, -58.4455137
user_point = Point(lon, lat)
user_point_gdf = gpd.GeoDataFrame(geometry=[user_point], crs="EPSG:4326")
user_point_utm = user_point_gdf.to_crs("EPSG:32721")
x, y = user_point_utm.geometry[0].x, user_point_utm.geometry[0].y
sample_box = box(x - 100, y - 100, x + 100, y + 100)
sample_box_gdf = gpd.GeoDataFrame(geometry=[sample_box], crs="EPSG:32721").to_crs("EPSG:4326")

# Filter data to just the sample area
sample_cells = grid[grid.intersects(sample_box_gdf.geometry[0])].copy()
sample_buildings = buildings[buildings.intersects(sample_box_gdf.geometry[0])].copy()

# Create map
m = folium.Map(location=[lat, lon], zoom_start=18, tiles='CartoDB positron')

# Add satellite option
folium.TileLayer(
    'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Imagery',
    name='Satellite'
).add_to(m)

# Add sample area outline
folium.GeoJson(
    sample_box_gdf.to_json(),
    style_function=lambda x: {'color': 'red', 'weight': 2, 'fillOpacity': 0},
    name="Sample Area (200×200m)"
).add_to(m)

# Add grid cells colored by density
folium.GeoJson(
    sample_cells.to_json(),
    style_function=lambda x: {
        'fillColor': '#1f78b4',
        'color': 'gray',
        'weight': 0.5,
        'fillOpacity': x['properties']['density'] * 0.7 if x['properties']['density'] else 0.1
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['density', 'avg_orientation', 'building_count'],
        aliases=['Density', 'Orientation (°)', 'Buildings'],
        localize=True
    ),
    name="Building Density"
).add_to(m)

# Add buildings
folium.GeoJson(
    sample_buildings.to_json(),
    style_function=lambda x: {'fillColor': 'red', 'color': 'red', 'weight': 1, 'fillOpacity': 0.5},
    name="Buildings"
).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Save and display map
m.save("sample_morphology_test.html")
m